# Table of Contents

- [0. Imports](#0-imports)
    - [0.1. Package and Libraries](#01-packages-and-libraries)
    - [0.2. Data](#02-data)
- [1. Base Tables Updates](#1-base-table-updates)
    - [1.1. Taxonomy](#11-taxonomy)
    - [1.2. Realms](#12-realms)
    - [1.3. References](#13-references)
- [2. Data Info](#2-data-info)
    - [2.1. Records](#21-records)
    - [2.2. Native Data](#22-native-data)
- [3. Tables Preparation](#3-tables-preparation)
    - [3.1. Native Data](#31-native-data)
        - [3.1.1. Update Table with Keys](#311-update-table-with-keys)
            - [3.1.1.1. Realm](#3111-realm)
            - [3.1.1.2. Taxonomy](#3112-taxonomy)
            - [3.1.1.3. References](#3113-references)
        - [3.1.2. Reorder Columns](#312-reorder-columns)
    - [3.2. Records Data](#32-records-data)
        - [3.2.1. Update Table with Keys](#321-update-table-with-keys)
            - [3.2.1.1. Area - Geographical Regions](#3211-area---geographical-regions)
            - [3.2.1.2. Realm](#3212-realm)
            - [3.2.1.3. Taxonomy](#3213-taxonomy)
            - [3.2.1.4. References](#3214-references)
        - [3.2.2. Data Filter](#322-data-filter)
            - [3.2.2.1. Drop Non-Introduced Species](#3221-drop-non-introduced-species)
        - [3.2.3. Reorder Columns](#321-reorder-columns)
- [4. Confirm Tables](#4-confirm-tables)
    - [4.1. Taxonomy](#41-taxonomy)
    - [4.2. Area - Geographical Region](#42-area---geographical-region)
    - [4.3. Realms](#43-realms)
    - [4.4. References](#44-references)
    - [4.5. Natives](#45-natives)
    - [4.6. Records](#46-records)
- [5. Database Tables Exportation](#5-database-tables-exportation)
- [6. SQLite3 Database Creation](#6-sqlite3-database-creation)
    - [6.1. Create Connection](#61-create-connection)
    - [6.2. Taxonomy](#62-taxonomy)
    - [6.3. Area - Geographical Region](#63-area---geographical-region)
    - [6.4. Realms](#64-realms)
    - [6.5. References](#65-references)
    - [6.6. Natives](#66-natives)
    - [6.7. Records](#67-records)
    - [6.8. Close Connector - Database Conclusion](#68-close-connector---database-conclusion)


# [0. Imports](#table-of-contents)

## [0.1. Packages and Libraries](#table-of-contents)

The packages used were:

>*pandas*: used to make all the tables into predefined schemas that can be imported for a relational database.

>*sqlite3*: we opted to use SQLite to store the database as this allows to have the database ready for analysis, easily stored and in a portable file that can be interpreted through R and Python.

>*os*: to rename the extension of the sqlite file from .db to .sqlite for a better interpretation on the outputs.

In [272]:
import pandas as pd
import sqlite3
import os

## [0.2. Data](#table-of-contents)

Importing the data that was already ready (Regions) or pre-prepared with DataCleaning.ipynb (RecData and Natives) or with ExtractTax_Ref (TaxonomyData and References)

>*Regions*: the predefined dataset that comprehends the selected regions for the register of each record - roughly corresponding to the administrative areas by continent and insularity. 

>*RecData*: the records dataset after cleaning, having the information on where each species was recorded, the arrival, and establishment. 

>*Natives*: the dataset with the native distribution of each species recorded as human mediated introduced and as established.

>*TaxonomyData*: the taxonomy dataset with all the species listed on *RecData*, their accepted name according to the methodology described in DataCleaning.ipynb, their genus and their family as retrieved and confirmed by the methodology on ExtractTax_Ref.ipynb.

>*References*: the reference list as extracted in ExtractTax_Ref.ipynb and the respective year.

In [273]:
Regions = pd.read_csv(r'../Data Raw/RegionsTableData.csv', sep=';', encoding='utf-8')
Regions.drop_duplicates(inplace=True)

RecData = pd.read_csv(r'../Transformed Data/RecordsClean.csv', sep=';', encoding='utf-8')
RecData['Species'] = RecData['Species'].apply(lambda x: ' '.join(x.split()[:2]))

Natives = pd.read_csv(r'../Transformed Data/NativeDataClean.csv', sep=';', encoding='utf-8')

TaxonomyData = pd.read_csv(r'../Transformed Data/TaxonomyClean.csv', sep=';', encoding='utf-8')

References = pd.read_csv(r'../Transformed Data/ReferencesClean.csv', sep=';', encoding='utf-8')

In [274]:
Regions.head(5)

,AreaID,AreaName,Country,Continent,SubContinent
0,ATA.1_1,Antarctica,Antarctica,Antarctica,NaN
1,NOR.2_1,Norway Antactic Islands,Norway,Antarctica,NaN
2,FRA.4_1,French Antarctic Islands,France,Antarctica,NaN
3,AUS.4_1,Australian Antarctic Territories,Australia,Antarctica,NaN
4,GBR.2_1,British Antarctic Islands,United Kingdom,Antarctica,NaN


In [275]:
RecData.head(5)

,Species,NAME_0,Realm,Cryptogenic,Dispersal,Eradicated,IntentionalRelease,Introduced,Established,ReportedFirstYear,Reference,ReferenceYear,AcceptedSpecies
0,Ostrinia nubilalis,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Ostrinia nubilalis
1,Zeiraphera diniana,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Zeiraphera griseana
2,Plutella xylostella,Canada,Nearctic,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Plutella xylostella
3,Helicoverpa armigera,Brazil,Neotropical,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Helicoverpa armigera
4,Spodoptera exempta,Seychelles,Oceanina,NaN,NaN,0.0,NaN,NaN,1.0,NaN,CAB International (CABI) (2024). CABI Invasive...,2024,Spodoptera exempta


In [276]:
Natives.head(5)

,Cosmopolitan,Continent,Species,Realm,ReferenceYear,Reference,AcceptedSpecies
0,0,Asia,Chilo infuscatellus,Saharo-Arabian,2024,"Li, A., et al. (2024). Sugarcane borers: speci...",Chilo infuscatellus
1,0,Asia,Actias selene,Saharo-Arabian,2007,"Choldumrongkul, S., et al. (2007). Geographica...",Actias selene
2,0,Asia,Caloptilia roscipennella,Saharo-Arabian,2024,"de Prins, J., de Prins, W. (2024). Global Taxo...",Caloptilia roscipennella
3,0,Asia,Elophila nymphaeata,Saharo-Arabian,1984,"Speidel, W. (1984). Revision der Acentropinae ...",Elophila nymphaeata
4,0,Asia,Papilio demoleus,Saharo-Arabian,2020,"Riaz, S., et al. (2020). Morphology, life cycl...",Papilio demoleus


In [277]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family
0,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae
1,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae
2,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae
3,Helicoverpa armigera,Helicoverpa armigera,Helicoverpa,Noctuidae
4,Spodoptera exempta,Spodoptera exempta,Spodoptera,Noctuidae


In [278]:
References.head(5)

,Reference,Reference Year
0,"Li, A., et al. (2024). Sugarcane borers: speci...",2024
1,"Choldumrongkul, S., et al. (2007). Geographica...",2007
2,"de Prins, J., de Prins, W. (2024). Global Taxo...",2024
3,"Speidel, W. (1984). Revision der Acentropinae ...",1984
4,"Riaz, S., et al. (2020). Morphology, life cycl...",2020


# [1. Base Table Updates](#table-of-contents)

## [1.1. Taxonomy](#table-of-contents)

*TaxonomyData* was almost ready, however it needed to have a Species Id that would allow to connect between the species on the Records dataset and the Taxonomy dataset. This Species Id was needed as well to connect inside the table between Species and the Accepted Species entry. 

To do so two separate columns were added: SpeciesID and AcceptedSpeciesID - to connect between the different datasets and in case of future updates to easily insert and correct for the new accepted names. 

In [279]:
TaxonomyData['SpeciesID'] = 'SP' + (TaxonomyData.index +1).astype(str) 
# Creating the SpeciesID column, which will be used as the primary key for the Taxonomy table. 
# This code will make the IDs to start with SP1 and complete with the number of rows.

In [280]:
species_to_id = dict(zip(TaxonomyData['Species'], TaxonomyData['SpeciesID'])) 
# Creating a dictionary with Species as key and SpeciesID 
# as value that will be used to map the SpeciesID column to the AcceptedSpecies column.

TaxonomyData['AcceptedSpeciesID'] = TaxonomyData['AcceptedSpecies'].map(species_to_id)

In [281]:
TaxonomyData.head(5)

,Species,AcceptedSpecies,Genus,Family,SpeciesID,AcceptedSpeciesID
0,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae,SP1,SP1
1,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae,SP2,SP1003
2,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae,SP3,SP3
3,Helicoverpa armigera,Helicoverpa armigera,Helicoverpa,Noctuidae,SP4,SP4
4,Spodoptera exempta,Spodoptera exempta,Spodoptera,Noctuidae,SP5,SP5


In [282]:
TaxonomyData = TaxonomyData[['SpeciesID', 'AcceptedSpeciesID', 'Species', 'AcceptedSpecies', 'Genus', 'Family']] #Reordering the columns to have a better readability

## [1.2. Realms](#table-of-contents)

In [283]:
RealmObs = RecData[['Realm']].copy()
RealmNat = Natives[['Realm']].copy()

Realms = pd.concat([RealmObs, RealmNat]).drop_duplicates().reset_index(drop=True).dropna() 
# Concatenating the Realms dataframes and removing the duplicates to keep all unique values for the Realms.
Realms['RealmID'] = 'RLM' + (Realms.index +1).astype(str) 
# Creating the RealmID column, which will be used as the primary key for the Realms table. This code will make the IDs to start with RLM1 and complete with the number of rows.

In [284]:
Realms = Realms[['RealmID', 'Realm']] 
# Reordering the columns to have a better readability

## [1.3. References](#table-of-contents)

In [285]:
References.head(5)

,Reference,Reference Year
0,"Li, A., et al. (2024). Sugarcane borers: speci...",2024
1,"Choldumrongkul, S., et al. (2007). Geographica...",2007
2,"de Prins, J., de Prins, W. (2024). Global Taxo...",2024
3,"Speidel, W. (1984). Revision der Acentropinae ...",1984
4,"Riaz, S., et al. (2020). Morphology, life cycl...",2020


In [286]:
References['ReferenceID'] = 'REF' + (References.index +1).astype(str) 
# Creating the ReferenceID column, which will be used as the primary key for the References table. 
# This code will make the IDs to start with REF1 and complete with the number of rows.

In [287]:
References.rename(columns={'Reference Year': 'ReferenceYear', 'Reference': 'BibliographicReference'}, inplace=True) 
# Renaming the Reference Year column to ReferenceYear and the Reference column to BibliographicReference for a better readability and be least prone to errors

In [288]:
References = References[['ReferenceID', 'BibliographicReference', 'ReferenceYear']] 
#Reordering the columns to have a better readability

# [2. Data Info](#table-of-contents)

A brief description of the data for the Records and Native distribution

## [2.1. Records](#table-of-contents)

In [289]:
RecData.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17445 entries, 0 to 17444
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Species             17445 non-null  object 
 1   NAME_0              17445 non-null  object 
 2   Realm               17445 non-null  object 
 3   Cryptogenic         4587 non-null   float64
 4   Dispersal           653 non-null    float64
 5   Eradicated          17401 non-null  float64
 6   IntentionalRelease  4653 non-null   float64
 7   Introduced          7136 non-null   float64
 8   Established         17255 non-null  float64
 9   ReportedFirstYear   2994 non-null   float64
 10  Reference           17445 non-null  object 
 11  ReferenceYear       17445 non-null  int64  
 12  AcceptedSpecies     17445 non-null  object 
dtypes: float64(7), int64(1), object(5)
memory usage: 1.7+ MB


In [290]:
RecData.describe().T

,count,mean,std,min,25%,50%,75%,max
Cryptogenic,4587.0,0.146283,0.353428,0.0,0.0,0.0,0.0,1.0
Dispersal,653.0,0.698315,0.459341,0.0,0.0,1.0,1.0,1.0
Eradicated,17401.0,0.004540,0.067228,0.0,0.0,0.0,0.0,1.0
IntentionalRelease,4653.0,0.054374,0.226778,0.0,0.0,0.0,0.0,1.0
Introduced,7136.0,0.970291,0.169794,0.0,1.0,1.0,1.0,1.0
Established,17255.0,0.968473,0.174742,0.0,1.0,1.0,1.0,1.0
ReportedFirstYear,2994.0,1976.954910,50.431234,1565.0,1962.0,1994.0,2009.0,2024.0
ReferenceYear,17445.0,2020.821496,6.215518,1926.0,2020.0,2024.0,2024.0,2025.0


In [291]:
RecData.describe(include = 'O').T

,count,unique,top,freq
Species,17445,1400,Spodoptera frugiperda,525
NAME_0,17445,232,United States,1804
Realm,17445,11,Palearctic,6902
Reference,17445,492,European and Mediterranean Plant Protection Or...,5104
AcceptedSpecies,17445,1257,Spodoptera frugiperda,525


In [292]:
RecData.columns

Index(['Species', 'NAME_0', 'Realm', 'Cryptogenic', 'Dispersal', 'Eradicated',
       'IntentionalRelease', 'Introduced', 'Established', 'ReportedFirstYear',
       'Reference', 'ReferenceYear', 'AcceptedSpecies'],
      dtype='object')

## [2.2. Native Data](#table-of-contents)

In [293]:
Natives.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2769 entries, 0 to 2768
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Cosmopolitan     2769 non-null   int64 
 1   Continent        2758 non-null   object
 2   Species          2769 non-null   object
 3   Realm            2762 non-null   object
 4   ReferenceYear    2769 non-null   int64 
 5   Reference        2769 non-null   object
 6   AcceptedSpecies  2769 non-null   object
dtypes: int64(2), object(5)
memory usage: 151.6+ KB


In [294]:
Natives.describe().T

,count,mean,std,min,25%,50%,75%,max
Cosmopolitan,2769.0,0.005417,0.073415,0.0,0.0,0.0,0.0,1.0
ReferenceYear,2769.0,2006.507403,21.352364,1775.0,2003.0,2012.0,2020.0,2025.0


In [295]:
Natives.describe(include = 'O').T

,count,unique,top,freq
Continent,2758,7,Asia,974
Species,2769,1034,Lampides boeticus,15
Realm,2762,11,Palearctic,812
Reference,2769,780,"Khramov, P. (Ed.) (2007). Insecta.pro: interna...",156
AcceptedSpecies,2769,1034,Lampides boeticus,15


In [296]:
Natives.columns

Index(['Cosmopolitan', 'Continent', 'Species', 'Realm', 'ReferenceYear',
       'Reference', 'AcceptedSpecies'],
      dtype='object')

# [3. Tables Preparation](#table-of-contents)

After the preparation on the base tables (Area, Taxonomy, Realms and References) it needs to be attributed the respective keys to the values on each data table (Native Distribution and Records). The columns should also be organized in the order for the sql import.

## [3.1. Native Data](#table-of-contents)

In [297]:
Natives

,Cosmopolitan,Continent,Species,Realm,ReferenceYear,Reference,AcceptedSpecies
0,0,Asia,Chilo infuscatellus,Saharo-Arabian,2024,"Li, A., et al. (2024). Sugarcane borers: speci...",Chilo infuscatellus
1,0,Asia,Actias selene,Saharo-Arabian,2007,"Choldumrongkul, S., et al. (2007). Geographica...",Actias selene
2,0,Asia,Caloptilia roscipennella,Saharo-Arabian,2024,"de Prins, J., de Prins, W. (2024). Global Taxo...",Caloptilia roscipennella
3,0,Asia,Elophila nymphaeata,Saharo-Arabian,1984,"Speidel, W. (1984). Revision der Acentropinae ...",Elophila nymphaeata
4,0,Asia,Papilio demoleus,Saharo-Arabian,2020,"Riaz, S., et al. (2020). Morphology, life cycl...",Papilio demoleus
...,...,...,...,...,...,...,...
2764,0,Asia,Parapoynx diminutalis,Saharo-Arabian,1996,"Buckingham, G., Bennett, C. (1996). Laboratory...",Parapoynx diminutalis
2765,0,Asia,Parapoynx diminutalis,Palearctic,1996,"Buckingham, G., Bennett, C. (1996). Laboratory...",Parapoynx diminutalis
2766,0,Africa,Sphenarches caffer,Madagascan,1910,"Fletcher, T. (1910). Lepidoptera exclusive of ...",Sphenarches caffer
2767,0,Asia,Sphenarches caffer,Oriental,1910,"Fletcher, T. (1910). Lepidoptera exclusive of ...",Sphenarches caffer


In [298]:
Natives_DB = Natives[['Species', 'Continent', 'Realm', 'Cosmopolitan', 'Reference']].copy() 
#Create a copy of the dataframe, organized and with the relevant columns, avoiding to repeat information

Natives_DB.rename(columns={'Reference': 'BibliographicReference'}, inplace=True) 
#Rename the column for merging and better readability

### [3.1.1. Update Table with Keys](#table-of-contents)

Attributing the respective IDs to be used as Foreign Keys in SQL

#### [3.1.1.1. Realm](#table-of-contents)

In [299]:
Natives_DB = Natives_DB.merge(Realms, on='Realm', how='left') #Merge with the Realms dataframe to attribute the respective RealmID
Natives_DB.drop(columns='Realm', inplace=True) #Drop the Realm column keeping the RealmID

#### [3.1.1.2. Taxonomy](#table-of-contents)

In [300]:
Natives_DB = Natives_DB.merge(TaxonomyData, on='Species', how='left')
#Merge with the Taxonomy dataframe to attribute the respective SpeciesID

Natives_DB.drop(columns=['Species', 'AcceptedSpecies', 'Genus', 'Family', 'AcceptedSpeciesID'], inplace=True)
#Drop the Species column and respective information keeping only the SpeciesID

#### [3.1.1.3. References](#table-of-contents)

In [301]:
Natives_DB = Natives_DB.merge(References, on='BibliographicReference', how='left')
#Merge with the References dataframe to attribute the respective ReferenceID

Natives_DB.drop(columns=['BibliographicReference', 'ReferenceYear'], inplace=True)
#Drop the BibliographicReference column and respective year keeping only the ReferenceID

### [3.1.2. Reorder Columns](#table-of-contents)

In [302]:
Natives_DB = Natives_DB[['SpeciesID', 'RealmID', 'Continent', 'Cosmopolitan', 'ReferenceID']]
#Reorder the columns to have the same order as it will be in the final database

## [3.2. Records Data](#table-of-contents)

In [303]:
RecData.rename(columns={'ReportedFirstYear': 'Year', #Rename the column for better readability
                        'NAME_0': 'Area', #Rename the column for merging and better readability
                        'Reference': 'BibliographicReference'}, inplace=True) #Rename the column for merging and better readability

In [304]:
#Create a copy of the dataframe, organized and with the relevant columns for the merging and the final database

Records_DB = RecData[['Species', #Species information
                           'Area', 'Realm', #Geographical information
                           'Cryptogenic', 'Introduced', 'Dispersal', 'IntentionalRelease', 'Year', #Arrival information
                           'Established', 'Eradicated', #Establishment information
                           'BibliographicReference']].copy() #Reference information


### [3.2.1. Update Table With Keys](#table-of-contents)


#### [3.2.1.1. Area - Geographical Regions](#table-of-contents)

In [305]:
RegionsData = Regions[['AreaID', 'AreaName', 'Country', 'Continent', 'SubContinent']].copy()

In [306]:
Records_DB = Records_DB.merge(RegionsData, left_on='Area', right_on='AreaName', how='left') 
# Merge with the Regions dataframe to attribute the respective AreaID

Records_DB.drop(columns=['AreaName', 'Area', 'Country', 'Continent', 'SubContinent'], inplace=True) 
# Drop the AreaName column and respective information keeping only the AreaID

#### [3.2.1.2. Realm](#table-of-contents)

In [307]:
Records_DB = Records_DB.merge(Realms, on='Realm', how='left') 
# Merge with the Realms dataframe to attribute the respective RealmID

Records_DB.drop(columns='Realm', inplace=True) 
# Drop the Realm column keeping the RealmID

#### [3.2.1.3. Taxonomy](#table-of-contents)

In [308]:
Records_DB = Records_DB.merge(TaxonomyData, on='Species', how='left') 
# Merge with the Taxonomy dataframe to attribute the respective SpeciesID

Records_DB.drop(columns=['Species', 'AcceptedSpecies', 'Genus', 'Family', 'AcceptedSpeciesID'], inplace=True) 
# Drop the Species column and respective information keeping only the SpeciesID

#### [3.2.1.4. References](#table-of-contents)

In [309]:
Records_DB = Records_DB.merge(References, on='BibliographicReference', how='left') 
# Merge with the References dataframe to attribute the respective ReferenceID

Records_DB.drop(columns=['BibliographicReference', 'ReferenceYear'], inplace=True) 
# Drop the BibliographicReference column and respective year keeping only the ReferenceID

### [3.2.2. Data Filter](#table-of-contents)

#### [3.2.2.1. Drop Non-introduced Species](#table-of-contents)

In [310]:
Records_DB = Records_DB.merge(TaxonomyData, on='SpeciesID', how='left')
# Merge with TaxonomyData to filter for Accepted Species

IntroducedSpecies = Records_DB[Records_DB['Introduced'] == 1].copy()
# Create a copy of the dataframe, keeping only the cases where Introduced == 1

Records_DB = Records_DB[Records_DB['AcceptedSpeciesID'].isin(IntroducedSpecies['AcceptedSpeciesID'])] 
# From Records_DB keep only the cases where AcceptedSpeciesID is on IntroducedSpecies


### [3.2.3. Reorder Columns](#table-of-contents)

In [311]:
#Reorder the columns to have the same order as it will be in the final database
Records_DB = Records_DB[['SpeciesID', #Species information
                         'AreaID', 'RealmID', #Geographical information
                                  'Cryptogenic', 'IntentionalRelease', 'Introduced', 'Dispersal', 'Established', 'Eradicated', 'Year',  #Invasion information
                                  'ReferenceID'] #Reference information
                        ].copy() 

In [312]:
Records_DB

,SpeciesID,AreaID,RealmID,Cryptogenic,IntentionalRelease,Introduced,Dispersal,Established,Eradicated,Year,ReferenceID
0,SP1,CAN.1_1,RLM1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,REF23
1,SP2,CAN.1_1,RLM1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,REF23
2,SP3,CAN.1_1,RLM1,NaN,NaN,NaN,NaN,1.0,0.0,NaN,REF23
3,SP4,BRA.1_1,RLM2,NaN,NaN,NaN,NaN,1.0,0.0,NaN,REF23
4,SP5,SYC.1_1,RLM3,NaN,NaN,NaN,NaN,1.0,0.0,NaN,REF23
...,...,...,...,...,...,...,...,...,...,...,...
17461,SP576,JPN.1_1,RLM5,0.0,0.0,1.0,0.0,1.0,0.0,NaN,REF780
17462,SP576,MDG.1_1,RLM11,0.0,0.0,1.0,0.0,1.0,0.0,NaN,REF780
17463,SP576,KOR.1_1,RLM7,0.0,0.0,1.0,0.0,1.0,0.0,NaN,REF780
17464,SP238,USA.1_1,RLM1,0.0,0.0,1.0,NaN,1.0,NaN,1976.0,REF15


In [313]:
#Converting the numerical columns to integer type
Records_DB['Cryptogenic'] = Records_DB['Cryptogenic'].astype('Int64')
Records_DB['IntentionalRelease'] = Records_DB['IntentionalRelease'].astype('Int64')
Records_DB['Introduced'] = Records_DB['Introduced'].astype('Int64')
Records_DB['Dispersal'] = Records_DB['Dispersal'].astype('Int64')
Records_DB['Established'] = Records_DB['Established'].astype('Int64')
Records_DB['Eradicated'] = Records_DB['Eradicated'].astype('Int64')
Records_DB['Year'] = Records_DB['Year'].astype('Int64')

# [4. Confirm Tables](#table-of-contents)

Final check for each table of the database before exporting each table to csv and importing them to the final database in sqlite.

## [4.1. Taxonomy](#table-of-contents)

In [314]:
TaxonomyData.head(5)

,SpeciesID,AcceptedSpeciesID,Species,AcceptedSpecies,Genus,Family
0,SP1,SP1,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae
1,SP2,SP1003,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae
2,SP3,SP3,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae
3,SP4,SP4,Helicoverpa armigera,Helicoverpa armigera,Helicoverpa,Noctuidae
4,SP5,SP5,Spodoptera exempta,Spodoptera exempta,Spodoptera,Noctuidae


In [315]:
TaxonomyData.columns

Index(['SpeciesID', 'AcceptedSpeciesID', 'Species', 'AcceptedSpecies', 'Genus',
       'Family'],
      dtype='object')

In [316]:
TaxonomyData.describe().T

,count,unique,top,freq
SpeciesID,1470,1470,SP1,1
AcceptedSpeciesID,1470,1257,SP1277,4
Species,1470,1470,Ostrinia nubilalis,1
AcceptedSpecies,1470,1257,Paraswammerdamia lutarea,4
Genus,1470,758,Coleophora,22
Family,1470,71,Tortricidae,186


In [317]:
TaxonomyData.describe(include = 'O').T

,count,unique,top,freq
SpeciesID,1470,1470,SP1,1
AcceptedSpeciesID,1470,1257,SP1277,4
Species,1470,1470,Ostrinia nubilalis,1
AcceptedSpecies,1470,1257,Paraswammerdamia lutarea,4
Genus,1470,758,Coleophora,22
Family,1470,71,Tortricidae,186


## [4.2. Area - Geographical Region](#table-of-contents)

In [318]:
RegionsData.head(5)

,AreaID,AreaName,Country,Continent,SubContinent
0,ATA.1_1,Antarctica,Antarctica,Antarctica,NaN
1,NOR.2_1,Norway Antactic Islands,Norway,Antarctica,NaN
2,FRA.4_1,French Antarctic Islands,France,Antarctica,NaN
3,AUS.4_1,Australian Antarctic Territories,Australia,Antarctica,NaN
4,GBR.2_1,British Antarctic Islands,United Kingdom,Antarctica,NaN


In [319]:
RegionsData.columns

Index(['AreaID', 'AreaName', 'Country', 'Continent', 'SubContinent'], dtype='object')

In [320]:
RegionsData.describe().T

,count,unique,top,freq
AreaID,258,258,ATA.1_1,1
AreaName,258,257,US Oceania Micronesia Islands,2
Country,258,205,France,8
Continent,258,8,Africa,64
SubContinent,22,3,Polynesia,9


In [321]:
RegionsData.describe(include = 'O').T

,count,unique,top,freq
AreaID,258,258,ATA.1_1,1
AreaName,258,257,US Oceania Micronesia Islands,2
Country,258,205,France,8
Continent,258,8,Africa,64
SubContinent,22,3,Polynesia,9


## [4.3. Realms](#table-of-contents)

In [322]:
Realms.head(5)

,RealmID,Realm
0,RLM1,Nearctic
1,RLM2,Neotropical
2,RLM3,Oceanina
3,RLM4,Oriental
4,RLM5,Sino-Japanese


In [323]:
Realms.columns

Index(['RealmID', 'Realm'], dtype='object')

In [324]:
Realms.describe().T

,count,unique,top,freq
RealmID,11,11,RLM1,1
Realm,11,11,Nearctic,1


In [325]:
Realms.describe(include = 'O').T

,count,unique,top,freq
RealmID,11,11,RLM1,1
Realm,11,11,Nearctic,1


## [4.4. References](#table-of-contents)

In [326]:
References.head(5)

,ReferenceID,BibliographicReference,ReferenceYear
0,REF1,"Li, A., et al. (2024). Sugarcane borers: speci...",2024
1,REF2,"Choldumrongkul, S., et al. (2007). Geographica...",2007
2,REF3,"de Prins, J., de Prins, W. (2024). Global Taxo...",2024
3,REF4,"Speidel, W. (1984). Revision der Acentropinae ...",1984
4,REF5,"Riaz, S., et al. (2020). Morphology, life cycl...",2020


In [327]:
References.columns

Index(['ReferenceID', 'BibliographicReference', 'ReferenceYear'], dtype='object')

In [328]:
References.describe().T

,count,mean,std,min,25%,50%,75%,max
ReferenceYear,1139.0,2004.532924,23.158989,1775.0,1999.0,2012.0,2019.0,2025.0


In [329]:
References.describe(include = 'O').T

,count,unique,top,freq
ReferenceID,1139,1139,REF1,1
BibliographicReference,1139,1138,"Zalucki, M., et al. (2007). Will biological co...",2


## [4.5. Natives](#table-of-contents)

In [330]:
Natives_DB.head(5)

,SpeciesID,RealmID,Continent,Cosmopolitan,ReferenceID
0,SP1028,RLM10,Asia,0,REF1
1,SP848,RLM10,Asia,0,REF2
2,SP181,RLM10,Asia,0,REF3
3,SP1093,RLM10,Asia,0,REF4
4,SP72,RLM10,Asia,0,REF5


In [331]:
Natives_DB.columns

Index(['SpeciesID', 'RealmID', 'Continent', 'Cosmopolitan', 'ReferenceID'], dtype='object')

In [332]:
Natives_DB.describe().T

,count,mean,std,min,25%,50%,75%,max
Cosmopolitan,2770.0,0.005415,0.073402,0.0,0.0,0.0,0.0,1.0


In [333]:
Natives_DB.describe(include = 'O').T

,count,unique,top,freq
SpeciesID,2770,1034,SP105,15
RealmID,2763,11,RLM7,812
Continent,2759,7,Asia,974
ReferenceID,2770,781,REF27,156


## [4.6. Observations](#table-of-contents)

In [334]:
Records_DB.head(5)

,SpeciesID,AreaID,RealmID,Cryptogenic,IntentionalRelease,Introduced,Dispersal,Established,Eradicated,Year,ReferenceID
0,SP1,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
1,SP2,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
2,SP3,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
3,SP4,BRA.1_1,RLM2,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
4,SP5,SYC.1_1,RLM3,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23


In [335]:
Records_DB.columns

Index(['SpeciesID', 'AreaID', 'RealmID', 'Cryptogenic', 'IntentionalRelease',
       'Introduced', 'Dispersal', 'Established', 'Eradicated', 'Year',
       'ReferenceID'],
      dtype='object')

In [336]:
Records_DB.describe().T

,count,mean,std,min,25%,50%,75%,max
Cryptogenic,4296.0,0.123371,0.3289,0.0,0.0,0.0,0.0,1.0
IntentionalRelease,4348.0,0.058188,0.234125,0.0,0.0,0.0,0.0,1.0
Introduced,6951.0,0.996691,0.057432,0.0,1.0,1.0,1.0,1.0
Dispersal,541.0,0.637708,0.481107,0.0,0.0,1.0,1.0,1.0
Established,15968.0,0.973948,0.159295,0.0,1.0,1.0,1.0,1.0
Eradicated,16030.0,0.004866,0.069588,0.0,0.0,0.0,0.0,1.0
Year,2743.0,1974.996719,51.722944,1565.0,1958.0,1992.0,2008.0,2024.0


In [337]:
Records_DB.describe(include = 'O').T

,count,unique,top,freq
SpeciesID,16073,1132,SP14,525
AreaID,16067,232,USA.1_1,1557
RealmID,16073,11,RLM7,6435
ReferenceID,16073,474,REF23,4616


# [5. Database Tables Exportation](#table-of-contents)

Export of each table to csv.

In [338]:
TaxonomyData.to_csv(r'../Database Tables/Base_Taxonomy.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')
RegionsData.to_csv(r'../Database Tables/Geography_Regions.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')
Realms.to_csv(r'../Database Tables/Geography_Realms.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')
References.to_csv(r'../Database Tables/Base_References.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')
Natives_DB.to_csv(r'../Database Tables/Obs_NativesDB.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')
Records_DB.to_csv(r'../Database Tables/Obs_Records_DB.csv', index=False, sep=";", encoding='utf-8', lineterminator='\n')

# [6. SQLite3 Database Creation](#table-of-contents)

## [6.1. Create Connection](#table-of-contents)

In [339]:
con = sqlite3.connect("../Database/WDNnL.db") 
# Create a connector to the database file or a file if it doesn't exist

con.execute("PRAGMA foreign_keys = ON") 
# Enable foreign keys in the new database

In [340]:
cur = con.cursor() 
# Create a cursor to execute SQL commands

In [341]:
tables_check = cur.execute("SELECT name FROM sqlite_master") 
# SQL code to check the existing tables in the database

tables_check.fetchall() 
# Execute the SQL code

[]

## [6.2. Taxonomy](#table-of-contents)

In [342]:
TaxonomyData

,SpeciesID,AcceptedSpeciesID,Species,AcceptedSpecies,Genus,Family
0,SP1,SP1,Ostrinia nubilalis,Ostrinia nubilalis,Ostrinia,Crambidae
1,SP2,SP1003,Zeiraphera diniana,Zeiraphera griseana,Zeiraphera,Tortricidae
2,SP3,SP3,Plutella xylostella,Plutella xylostella,Plutella,Plutellidae
3,SP4,SP4,Helicoverpa armigera,Helicoverpa armigera,Helicoverpa,Noctuidae
4,SP5,SP5,Spodoptera exempta,Spodoptera exempta,Spodoptera,Noctuidae
...,...,...,...,...,...,...
1465,SP1466,SP1466,Clethrogyna turbata,Clethrogyna turbata,Clethrogyna,Erebidae
1466,SP1467,SP1467,Elkalyce argiades,Elkalyce argiades,Elkalyce,Lycaenidae
1467,SP1468,SP1468,Zale minerea,Zale minerea,Zale,Erebidae
1468,SP1469,SP1469,Notioplusia illustrata,Notioplusia illustrata,Notioplusia,Noctuidae


In [343]:
TaxonomyData = TaxonomyData.reset_index(drop=True) 
# Reset the index of the Taxonomy dataframe

In [344]:
cur.execute("""CREATE TABLE IF NOT EXISTS Taxonomy (
    SpeciesID VARCHAR(7) PRIMARY KEY, 
    AcceptedSpeciesID VARCHAR(7),
    Species TEXT, 
    AcceptedSpecies TEXT, 
    Genus TEXT, 
    Family TEXT,
    FOREIGN KEY (AcceptedSpeciesID) REFERENCES Taxonomy(SpeciesID) ON DELETE SET NULL)
    """)

#SQL code to create the Taxonomy table, using the SpeciesID as the primary key and the AcceptedSpeciesID as a foreign key.
#The types of each column are also defined as VARCHAR(7) for SpeciesID and AcceptedSpeciesID and TEXT for the remaining columns

In [345]:
cols = ['SpeciesID', 'AcceptedSpeciesID', 'Species', 'AcceptedSpecies', 'Genus', 'Family'] 
# Define the columns to be selected and inserted into the database

In [346]:
BaseTaxonomy = TaxonomyData[TaxonomyData['SpeciesID'] == TaxonomyData['AcceptedSpeciesID']] 
# Select only the rows where the SpeciesID is equal to the AcceptedSpeciesID

BaseTaxonomy = BaseTaxonomy[cols] 
# Select only the columns defined in the cols variable

BaseTaxonomy.to_sql('Taxonomy', con, if_exists='append', index=False) 
# Insert the selected rows into the Taxonomy table in the database

con.commit() 
# Commit the changes

In [347]:
ComposedTaxonomy = TaxonomyData[TaxonomyData['SpeciesID'] != TaxonomyData['AcceptedSpeciesID']] 
# Select only the rows where the SpeciesID is not equal to the AcceptedSpeciesID

ComposedTaxonomy = ComposedTaxonomy[cols] 
# Select only the columns defined in the cols variable

ComposedTaxonomy.to_sql('Taxonomy', con, if_exists='append', index=False) 
# Insert the selected rows into the Taxonomy table in the database

con.commit() 
# Commit the changes

In [348]:
Taxonomy_check = cur.execute("SELECT * FROM Taxonomy") 
# SQL code to select all the rows from the Taxonomy table

Taxonomy_check.fetchone() 
# Execute the SQL code, fetching only the results for the first row

('SP1',
 'SP1',
 'Ostrinia nubilalis',
 'Ostrinia nubilalis',
 'Ostrinia',
 'Crambidae')

In [349]:
#cur.execute("""DROP TABLE Taxonomy""") 
# SQL code to drop the Taxonomy table, kept for testing purposes

## [6.3. Area - Geographical Region](#table-of-contents)

In [350]:
RegionsData = RegionsData[['AreaID', 'AreaName', 'Country', 'Continent', 'SubContinent']] 
# Select only the columns to be inserted into the database

RegionsData = RegionsData.reset_index(drop=True) 
# Reset the index of the Regions dataframe

In [351]:
cur.execute("""CREATE TABLE IF NOT EXISTS Regions (
    AreaID VARCHAR(8) PRIMARY KEY, 
    AreaName TEXT, 
    Country TEXT, 
    Continent TEXT, 
    SubContinent TEXT)
    """)
# SQL code to create the Regions table, using the AreaID as the primary key
# The types of each column are also defined as VARCHAR(8) for AreaID and TEXT for the remaining columns

In [352]:
RegionsData.to_sql('Regions', con, if_exists='append', index=False) 
# Insert the Regions data into the Regions table in the database

con.commit() 
# Commit the changes

In [353]:
Regions_check = cur.execute('SELECT * FROM "Regions"') 
# SQL code to select all the rows from the Regions table

Regions_check.fetchone() 
# Execute the previous SQL code, fetching only the results for the first row


('ATA.1_1', 'Antarctica', 'Antarctica', 'Antarctica', None)

In [354]:
#cur.execute("""DROP TABLE Regions""") 
# SQL code to drop the Regions table, kept for testing purposes

## [6.4. Realms](#table-of-contents)

In [355]:
Realms = Realms[['RealmID', 'Realm']] 
# Reorder and select the columns to be inserted into the database

Realms = Realms.reset_index(drop=True) 
# Reset the index of the Realms dataframe

In [356]:
cur.execute("""CREATE TABLE IF NOT EXISTS Realms (
    RealmID VARCHAR(5) PRIMARY KEY, 
    Realm TEXT)
    """)
# SQL code to create the Realms table, using the RealmID as the primary key
# The types of each column are also defined as VARCHAR(5) for RealmID and TEXT for the Realm column

In [357]:
Realms.to_sql('Realms', con, if_exists='append', index=False) 
# Insert the Realms data into the Realms table in the database

con.commit() 
# Commit the changes

In [358]:
Realms_check = cur.execute('SELECT * FROM "Realms"') 
# SQL code to select all the rows from the Realms table

Realms_check.fetchone() 
# Execute the previous SQL code, fetching only the results for the first row

('RLM1', 'Nearctic')

In [359]:
#cur.execute("""DROP TABLE Realms""") 
# SQL code to drop the Realms table, kept for testing purposes

## [6.5. References](#table-of-contents)

In [360]:
References = References.reset_index(drop=True) 
# Reset the index of the References dataframe

In [361]:
cur.execute("""CREATE TABLE IF NOT EXISTS "References" (
    ReferenceID VARCHAR(7) PRIMARY KEY, 
    BibliographicReference TEXT, 
    ReferenceYear INTEGER
)""")
# SQL code to create the References table, using the ReferenceID as the primary key
# The types of each column are also defined as VARCHAR(7) for ReferenceID, TEXT for BibliographicReference and INTEGER for ReferenceYear

In [362]:
References.to_sql('References', con, if_exists='append', index=False) 
# Insert the References data into the References table in the database

con.commit() 
# Commit the changes

In [363]:
References_check = cur.execute('SELECT * FROM "References"') 
# SQL code to select all the rows from the References table

References_check.fetchone() 
# Execute the previous SQL code, fetching only the results for the first row


('REF1',
 'Li, A., et al. (2024). Sugarcane borers: species, distribution, damage and management options. Journal of Pest Science. 97(3): 1171-1201',
 2024)

In [364]:
#cur.execute('DROP TABLE "References"') 
# SQL code to drop the References table, kept for testing purposes

## [6.6. Natives](#table-of-contents)

In [365]:
Natives_DB = Natives_DB.reset_index(drop=True) 
# Reset the index of the Natives dataframe

In [366]:
cur.execute("""
CREATE TABLE IF NOT EXISTS NativeDistribution (
    SpeciesID VARCHAR(7), 
    RealmID VARCHAR(5), 
    Continent TEXT,
    Cosmopolitan INTEGER CHECK (Cosmopolitan IN (0, 1) OR Cosmopolitan IS NULL),
    ReferenceID VARCHAR(7), 
    FOREIGN KEY (SpeciesID) REFERENCES Taxonomy(SpeciesID) ON DELETE SET NULL,
    FOREIGN KEY (RealmID) REFERENCES Realms(RealmID) ON DELETE SET NULL,
    FOREIGN KEY (ReferenceID) REFERENCES "References"(ReferenceID) ON DELETE SET NULL
)""")

# SQL code to create the NativeDistribution table, using the SpeciesID as the primary key and the RealmID and ReferenceID as foreign keys referring to the respective tables
# The types of each column are also defined as VARCHAR(7) for SpeciesID and ReferenceID, VARCHAR(5) for RealmID, TEXT for Continent and INTEGER for Cosmopolitan
# In case the Cosmopolitan column has a value other than 0 or 1, the value will be set to NULL
# In case the ReferenceID column does not exist in the References table, the value will be set to NULL
# In case the RealmID column does not exist in the Realms table, the value will be set to NULL
# In case the SpeciesID column does not exist in the Taxonomy table, the value will be set to NULL 

In [367]:
Natives_DB.to_sql('NativeDistribution', con, if_exists='append', index=False) 
# Insert the Natives data into the NativeDistribution table in the database

con.commit() 
# Commit the changes

In [368]:
NativeDist_check = cur.execute('SELECT * FROM NativeDistribution') 
# SQL code to select all the rows from the NativeDistribution table

NativeDist_check.fetchone() 
# Execute the previous SQL code, fetching only the results for the first row

('SP1028', 'RLM10', 'Asia', 0, 'REF1')

In [369]:
#cur.execute('DROP TABLE NativeDistribution') 
# SQL code to drop the NativeDistribution table, kept for testing purposes

## [6.7. Records](#table-of-contents)

In [370]:
Records_DB = Records_DB.reset_index(drop=True) 
# Reset the index of the Records dataframe

In [371]:
Records_DB

,SpeciesID,AreaID,RealmID,Cryptogenic,IntentionalRelease,Introduced,Dispersal,Established,Eradicated,Year,ReferenceID
0,SP1,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
1,SP2,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
2,SP3,CAN.1_1,RLM1,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
3,SP4,BRA.1_1,RLM2,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
4,SP5,SYC.1_1,RLM3,<NA>,<NA>,<NA>,<NA>,1,0,<NA>,REF23
...,...,...,...,...,...,...,...,...,...,...,...
16068,SP576,JPN.1_1,RLM5,0,0,1,0,1,0,<NA>,REF780
16069,SP576,MDG.1_1,RLM11,0,0,1,0,1,0,<NA>,REF780
16070,SP576,KOR.1_1,RLM7,0,0,1,0,1,0,<NA>,REF780
16071,SP238,USA.1_1,RLM1,0,0,1,<NA>,1,<NA>,1976,REF15


In [372]:
cur.execute("""
CREATE TABLE IF NOT EXISTS Records (
    SpeciesID VARCHAR(7),
    AreaID VARCHAR(8), 
    RealmID VARCHAR(5), 
    Cryptogenic INTEGER CHECK (Cryptogenic IN (0, 1) OR Cryptogenic IS NULL),
    IntentionalRelease INTEGER CHECK (Cryptogenic IN (0, 1) OR Cryptogenic IS NULL),
    Introduced INTEGER CHECK (Introduced IN (0, 1) OR Introduced IS NULL),
    Dispersal INTEGER CHECK (Dispersal IN (0, 1) OR Dispersal IS NULL),
    Established INTEGER CHECK (Established IN (0, 1) OR Established IS NULL),
    Eradicated INTEGER CHECK (Eradicated IN (0, 1) OR Eradicated IS NULL),
    Year INTEGER,
    ReferenceID VARCHAR(7),
    FOREIGN KEY (SpeciesID) REFERENCES Taxonomy(SpeciesID) ON DELETE SET NULL,
    FOREIGN KEY (AreaID) REFERENCES Regions(AreaID) ON DELETE SET NULL,
    FOREIGN KEY (RealmID) REFERENCES Realms(RealmID) ON DELETE SET NULL,
    FOREIGN KEY (ReferenceID) REFERENCES "References"(ReferenceID) ON DELETE SET NULL
)""")

# SQL code to create the Records table, using the SpeciesID as the primary key and the AreaID, RealmID and ReferenceID as foreign keys referring to the respective tables
# The types of each column are also defined as VARCHAR(7) for SpeciesID and ReferenceID, VARCHAR(8) for AreaID, VARCHAR(5) for RealmID, INTEGER for all other columns
# In case a column that has a foreign key does not exist in the respective table, the value will be set to NULL
# In case a non Year column that is an integer has a value other than 0 or 1, the value will be set to NULL

In [373]:
Records_DB.to_sql('Records', con, if_exists='append', index=False) 
# Insert the Records data into the Records table in the database

con.commit() 
# Commit the changes

In [374]:
Records_check = cur.execute('SELECT * FROM Records') 
# SQL code to select all the rows from the Records table

Records_check.fetchone() 
# Execute the previous SQL code, fetching only the results for the first row

('SP1', 'CAN.1_1', 'RLM1', None, None, None, None, 1, 0, None, 'REF23')

In [375]:
#cur.execute('DROP TABLE Records') 
# SQL code to drop the Records table, kept for testing purposes

## [6.8. Close Connector - Database Conclusion](#table-of-contents)

In [376]:
con.close() 
#Close the connection to the database

In [377]:
os.rename(r'../Database/WDNnL.db', r'../Database/WDNnL.sqlite') 
#Rename the WDNnL.db file to WDNnL.sqlite for better readability

In [378]:
con = sqlite3.connect("../Database/WDNnL.sqlite") 
#Connect to the WDNnL.sqlite database

cur = con.cursor() 
#Create a cursor to execute SQL commands

In [379]:
tables_check = cur.execute("SELECT name FROM sqlite_master") 
#SQL code to select all the tables from the database

tables_check.fetchall() 
#Execute the SQL code

[('Taxonomy',),
 ('sqlite_autoindex_Taxonomy_1',),
 ('Regions',),
 ('sqlite_autoindex_Regions_1',),
 ('Realms',),
 ('sqlite_autoindex_Realms_1',),
 ('References',),
 ('sqlite_autoindex_References_1',),
 ('NativeDistribution',),
 ('Records',)]

In [380]:
con.close() 
#Close the connection to the database